# 01 — Colab Setup

**One-time setup** cho Colab Pro environment. Chạy 1 lần để chuẩn bị infrastructure.

Phases:
1. GPU check + GDrive mount
2. Clone repo + install deps
3. Configure secrets (OpenAI + Neo4j Aura)
4. Push KG → Neo4j Aura
5. Smoke test 1 question end-to-end

Estimated time: 30 min.

## 1.1 GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 1.2 Clone repo + install deps

In [ ]:
import os
REPO_DIR = '/content/legal-graph-kb'
if not os.path.exists(REPO_DIR):
    # Option A: clone từ GitHub (nếu public)
    # !git clone https://github.com/your-user/legal-graph-kb.git $REPO_DIR
    # Option B: copy từ GDrive (nếu push code lên Drive)
    !cp -r /content/drive/MyDrive/legal-graph-kb $REPO_DIR
%cd $REPO_DIR
!ls

In [ ]:
# Install deps. Colab đã có nhiều — chỉ install những thứ thiếu
!pip install -q neo4j sentence-transformers openai python-dotenv bert-score
!pip install -q tqdm pandas numpy
# SWI-Prolog cho elite
!apt-get update -q && apt-get install -y -q swi-prolog
!swipl --version

## 1.3 Secrets — OpenAI + Neo4j Aura

Trong Colab UI → 🔑 icon → add secrets:
- `OPENAI_API_KEY`
- `NEO4J_URI` (Aura: `neo4j+s://xxx.databases.neo4j.io`)
- `NEO4J_USER` (Aura default: `neo4j`)
- `NEO4J_PASSWORD`

Mỗi secret enable cho notebook này.

In [ ]:
from google.colab import userdata
import os

for key in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    val = userdata.get(key)
    os.environ[key] = val
    print(f'{key}: {"set" if val else "MISSING"} ({len(val) if val else 0} chars)')
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'  # use T4 GPU

## 1.4 Verify Neo4j Aura connection

In [ ]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(
    os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USER'], os.environ['NEO4J_PASSWORD']),
)
with driver.session(database='neo4j') as s:
    result = s.run('CALL dbms.components() YIELD versions').single()
    print('Neo4j version:', result['versions'][0])
driver.close()

## 1.5 Push KG → Neo4j Aura

Load articles + clauses + embeddings từ local export.

In [ ]:
# Check KG data files có sẵn không
!ls -lh data/interim/structured_law.json data/processed/*.parquet 2>&1 | head

In [ ]:
# Push KG sử dụng existing load script. Nếu Aura đã có data → skip.
with driver.session(database='neo4j') as s:
    n_articles = s.run('MATCH (n:Article) RETURN count(n) AS c').single()['c']
    n_clauses = s.run('MATCH (n:Clause) RETURN count(n) AS c').single()['c']
    print(f'Existing Aura: {n_articles} Articles, {n_clauses} Clauses')

if n_articles == 0:
    !python -m src.load_neo4j
else:
    print('KG đã có sẵn, skip load.')

In [ ]:
# Tạo vector index nếu chưa có (cần cho retrieval)
with driver.session(database='neo4j') as s:
    indexes = s.run('SHOW INDEXES').data()
    has_vec = any('clause_vec' in (idx.get('name') or '') for idx in indexes)
    if not has_vec:
        s.run('''
        CREATE VECTOR INDEX clause_vec IF NOT EXISTS
        FOR (c:Clause) ON c.embedding
        OPTIONS {indexConfig: {`vector.dimensions`: 1024, `vector.similarity_function`: 'cosine'}}
        ''')
        print('Created clause_vec index')
    else:
        print('clause_vec index exists ✓')

## 1.6 Smoke test — 1 câu end-to-end

Chạy 1 question qua graphrag arm để verify toàn bộ pipeline.

In [ ]:
import sys
sys.path.insert(0, '/content/legal-graph-kb')
from src.rag_query import RagPipeline

pipeline = RagPipeline()
_ = pipeline.embed_model  # warm up BGE-M3 on T4
result = pipeline.ask('Tôi đã đóng BHXH 12 năm, có đủ điều kiện hưởng lương hưu không?',
                       top_k=8, verbose=False)
print('Answer:', result.answer[:300])
print('Citations:', result.citation_ids)
print('Elapsed:', result.elapsed_s, 's')
pipeline.close()

## 1.7 Smoke test — elite arm (Prolog)

In [ ]:
from experiments.elite_pipelines import EliteGraphRAGPipeline
p = EliteGraphRAGPipeline(model='gpt-4o-mini')
ans = p.ask('Tôi đã đóng BHXH 12 năm, có đủ điều kiện hưởng lương hưu không?')
print('Prolog success:', ans.prolog_success)
print('Status:', ans.prolog_status)
print('IRAC[:300]:', ans.answer[:300])
print('Plain answer:', ans.plain_answer)
p.close()

## 1.8 Sync output back to GDrive

Đảm bảo các result directories được persist sau khi session timeout.

In [ ]:
# Tạo symlink data/eval → GDrive cho persistent storage
import os
GDRIVE_BASE = '/content/drive/MyDrive/legal-graph-kb-results'
os.makedirs(f'{GDRIVE_BASE}/data/eval', exist_ok=True)
# Symlink (không copy) — write-through tới GDrive
!rm -rf /content/legal-graph-kb/data/eval
!ln -s {GDRIVE_BASE}/data/eval /content/legal-graph-kb/data/eval
!ls -la /content/legal-graph-kb/data/eval

## Done — Ready for inference

Move to `02_colab_inference.ipynb`.